# Data Preprocessing for Predictive Maintenance

## Objective

This notebook prepares the cleaned NASA C-MAPSS FD004 dataset for machine learning.

The preprocessing pipeline includes:

- Loading the cleaned dataset
- Creating the Remaining Useful Life (RUL) target
- Verifying the generated target
- Saving the dataset with RUL
- Feature Selection
- Feature Variance Analysis
- Feature Correlation Analysis
- Feature Scaling
- Saving the final processed dataset

In [1]:
# Data Manipulation
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Feature Scaling
from sklearn.preprocessing import StandardScaler, MinMaxScaler

# Ignore Warnings
import warnings
warnings.filterwarnings("ignore")

## Why Preprocessing?
Raw datasets cannot be used directly for machine learning.
Preprocessing transforms the cleaned dataset into a machine learning–ready dataset by creating the target variable (RUL), selecting useful features, analyzing feature importance, and scaling numerical features where appropriate.

In [2]:
train = pd.read_csv(r"D:\Project\aircraft maintenance system\data\processed\train_clean.csv", index_col=0)

train.head(10)

,cycle,setting1,setting2,setting3,s1,s2,s3,s4,s5,s6,...,s12,s13,s14,s15,s16,s17,s18,s19,s20,s21
unit,,,,,,,,,,,,,,,,,,,,,
1,1,42.0049,0.8400,100.0,445.00,549.68,1343.43,1112.93,3.91,5.70,...,129.78,2387.99,8074.83,9.3335,0.02,330,2212,100.00,10.62,6.3670
1,2,20.0020,0.7002,100.0,491.19,606.07,1477.61,1237.50,9.35,13.61,...,312.59,2387.73,8046.13,9.1913,0.02,361,2324,100.00,24.37,14.6552
1,3,42.0038,0.8409,100.0,445.00,548.95,1343.12,1117.05,3.91,5.69,...,129.62,2387.97,8066.62,9.4007,0.02,329,2212,100.00,10.48,6.4213
1,4,42.0000,0.8400,100.0,445.00,548.70,1341.24,1118.03,3.91,5.70,...,129.80,2388.02,8076.05,9.3369,0.02,328,2212,100.00,10.54,6.4176
1,5,25.0063,0.6207,60.0,462.54,536.10,1255.23,1033.59,7.05,9.00,...,164.11,2028.08,7865.80,10.8366,0.02,305,1915,84.93,14.03,8.6754
1,6,34.9996,0.8400,100.0,449.44,554.77,1352.87,1117.01,5.48,7.97,...,181.90,2387.87,8054.10,9.3346,0.02,330,2223,100.00,14.91,8.9057
1,7,0.0019,0.0001,100.0,518.67,641.83,1583.47,1393.89,14.62,21.58,...,520.48,2387.89,8127.92,8.3960,0.03,391,2388,100.00,38.93,23.4578
1,8,41.9981,0.8400,100.0,445.00,549.05,1344.16,1110.77,3.91,5.69,...,129.65,2387.97,8075.99,9.3679,0.02,329,2212,100.00,10.55,6.2787
1,9,42.0016,0.8400,100.0,445.00,549.55,1342.85,1101.67,3.91,5.70,...,129.65,2388.00,8071.13,9.3384,0.02,328,2212,100.00,10.63,6.3055


In [3]:
# Find the maximum operating cycle for each engine
max_cycle = train.groupby("unit")["cycle"].max().reset_index()
max_cycle.columns = ["unit", "max_cycle"]

# Display first few rows
max_cycle.head()

,unit,max_cycle
0,1,321
1,2,299
2,3,307
3,4,274
4,5,193


In [4]:

# Merge maximum cycle information with the original dataset
train = train.merge(max_cycle, on="unit")

train.head()

,unit,cycle,setting1,setting2,setting3,s1,s2,s3,s4,s5,...,s13,s14,s15,s16,s17,s18,s19,s20,s21,max_cycle
0,1,1,42.0049,0.8400,100.0,445.00,549.68,1343.43,1112.93,3.91,...,2387.99,8074.83,9.3335,0.02,330,2212,100.00,10.62,6.3670,321
1,1,2,20.0020,0.7002,100.0,491.19,606.07,1477.61,1237.50,9.35,...,2387.73,8046.13,9.1913,0.02,361,2324,100.00,24.37,14.6552,321
2,1,3,42.0038,0.8409,100.0,445.00,548.95,1343.12,1117.05,3.91,...,2387.97,8066.62,9.4007,0.02,329,2212,100.00,10.48,6.4213,321
3,1,4,42.0000,0.8400,100.0,445.00,548.70,1341.24,1118.03,3.91,...,2388.02,8076.05,9.3369,0.02,328,2212,100.00,10.54,6.4176,321
4,1,5,25.0063,0.6207,60.0,462.54,536.10,1255.23,1033.59,7.05,...,2028.08,7865.80,10.8366,0.02,305,1915,84.93,14.03,8.6754,321


# Step 1: Create Remaining Useful Life (RUL)

Remaining Useful Life (RUL) is the number of operating cycles remaining before an engine reaches failure.

Since the training dataset does not contain RUL values, they are computed using the maximum cycle reached by each engine.

In [5]:
# Calculate Remaining Useful Life (RUL)
train["RUL"] = train["max_cycle"] - train["cycle"]

train.head()

,unit,cycle,setting1,setting2,setting3,s1,s2,s3,s4,s5,...,s14,s15,s16,s17,s18,s19,s20,s21,max_cycle,RUL
0,1,1,42.0049,0.8400,100.0,445.00,549.68,1343.43,1112.93,3.91,...,8074.83,9.3335,0.02,330,2212,100.00,10.62,6.3670,321,320
1,1,2,20.0020,0.7002,100.0,491.19,606.07,1477.61,1237.50,9.35,...,8046.13,9.1913,0.02,361,2324,100.00,24.37,14.6552,321,319
2,1,3,42.0038,0.8409,100.0,445.00,548.95,1343.12,1117.05,3.91,...,8066.62,9.4007,0.02,329,2212,100.00,10.48,6.4213,321,318
3,1,4,42.0000,0.8400,100.0,445.00,548.70,1341.24,1118.03,3.91,...,8076.05,9.3369,0.02,328,2212,100.00,10.54,6.4176,321,317
4,1,5,25.0063,0.6207,60.0,462.54,536.10,1255.23,1033.59,7.05,...,7865.80,10.8366,0.02,305,1915,84.93,14.03,8.6754,321,316


In [6]:
# Remove the helper column
train.drop(columns="max_cycle", inplace=True)

train.head()

,unit,cycle,setting1,setting2,setting3,s1,s2,s3,s4,s5,...,s13,s14,s15,s16,s17,s18,s19,s20,s21,RUL
0,1,1,42.0049,0.8400,100.0,445.00,549.68,1343.43,1112.93,3.91,...,2387.99,8074.83,9.3335,0.02,330,2212,100.00,10.62,6.3670,320
1,1,2,20.0020,0.7002,100.0,491.19,606.07,1477.61,1237.50,9.35,...,2387.73,8046.13,9.1913,0.02,361,2324,100.00,24.37,14.6552,319
2,1,3,42.0038,0.8409,100.0,445.00,548.95,1343.12,1117.05,3.91,...,2387.97,8066.62,9.4007,0.02,329,2212,100.00,10.48,6.4213,318
3,1,4,42.0000,0.8400,100.0,445.00,548.70,1341.24,1118.03,3.91,...,2388.02,8076.05,9.3369,0.02,328,2212,100.00,10.54,6.4176,317
4,1,5,25.0063,0.6207,60.0,462.54,536.10,1255.23,1033.59,7.05,...,2028.08,7865.80,10.8366,0.02,305,1915,84.93,14.03,8.6754,316


In [7]:
# Check RUL statistics
train["RUL"].describe()

count    61249.000000
mean       133.311417
std         89.783389
min          0.000000
25%         61.000000
50%        122.000000
75%        190.000000
max        542.000000
Name: RUL, dtype: float64

## Step 2: Verify Generated RUL

After creating the target variable, it is important to verify that:

- no negative values exist
- maximum values are correct
- the generated RUL follows the expected distribution

In [8]:
# Check for negative RUL values
print("Negative RUL values:", (train["RUL"] < 0).sum())

Negative RUL values: 0


In [9]:
train.head()

,unit,cycle,setting1,setting2,setting3,s1,s2,s3,s4,s5,...,s13,s14,s15,s16,s17,s18,s19,s20,s21,RUL
0,1,1,42.0049,0.8400,100.0,445.00,549.68,1343.43,1112.93,3.91,...,2387.99,8074.83,9.3335,0.02,330,2212,100.00,10.62,6.3670,320
1,1,2,20.0020,0.7002,100.0,491.19,606.07,1477.61,1237.50,9.35,...,2387.73,8046.13,9.1913,0.02,361,2324,100.00,24.37,14.6552,319
2,1,3,42.0038,0.8409,100.0,445.00,548.95,1343.12,1117.05,3.91,...,2387.97,8066.62,9.4007,0.02,329,2212,100.00,10.48,6.4213,318
3,1,4,42.0000,0.8400,100.0,445.00,548.70,1341.24,1118.03,3.91,...,2388.02,8076.05,9.3369,0.02,328,2212,100.00,10.54,6.4176,317
4,1,5,25.0063,0.6207,60.0,462.54,536.10,1255.23,1033.59,7.05,...,2028.08,7865.80,10.8366,0.02,305,1915,84.93,14.03,8.6754,316


In [10]:
# Save dataset with RUL
train.to_csv(r"D:\Project\aircraft maintenance system\data\processed\train_rul.csv", index=False)

print("train_rul.csv saved successfully!")

train_rul.csv saved successfully!


## Step 3: Feature Selection

The dataset contains operational settings, sensor measurements, engine identifiers, operating cycles, and the target variable (RUL).

Before training a machine learning model, the input features (X) and target variable (y) are separated.

- `unit` is an engine identifier and does not provide useful predictive information.
- `RUL` is the target variable to be predicted.
- The remaining columns are used as input features for model training.

In [11]:
# Separate features and target

X = train.drop(columns=["unit", "RUL"])
y = train["RUL"]

print("Feature Matrix Shape :", X.shape)
print("Target Vector Shape  :", y.shape)

Feature Matrix Shape : (61249, 25)
Target Vector Shape  : (61249,)


In [12]:
X.columns.tolist()

['cycle',
 'setting1',
 'setting2',
 'setting3',
 's1',
 's2',
 's3',
 's4',
 's5',
 's6',
 's7',
 's8',
 's9',
 's10',
 's11',
 's12',
 's13',
 's14',
 's15',
 's16',
 's17',
 's18',
 's19',
 's20',
 's21']

In [13]:
y.head()

0    320
1    319
2    318
3    317
4    316
Name: RUL, dtype: int64

## Step 4: Feature Variance Analysis

Not all sensor measurements contribute equally to predicting Remaining Useful Life (RUL). Some sensors may remain nearly constant throughout the engine's operation and provide little useful information.

Variance analysis helps identify features with very low or zero variance, which can be considered for removal to improve model efficiency and reduce noise.

In [15]:
# Calculate variance of each feature
feature_variance = X.var().sort_values(ascending=False)
feature_variance

s9          113520.171620
s7           21573.795950
s18          21162.245693
s8           21126.111710
s12          19176.463675
s13          16434.691116
s4           14239.073899
s3           11271.558809
cycle         8061.057015
s14           7339.442015
s2            1394.473263
s17            773.300629
s1             698.906067
setting1       218.469733
setting3       203.118191
s20             98.731968
s21             35.553759
s6              29.637316
s19             28.830710
s5              13.125202
s11             10.520237
s15              0.563061
setting2         0.096537
s10              0.016302
s16              0.000022
dtype: float64

In [16]:
variance_df = pd.DataFrame({
    "Feature": feature_variance.index,
    "Variance": feature_variance.values
})

variance_df

,Feature,Variance
0,s9,113520.171620
1,s7,21573.795950
2,s18,21162.245693
3,s8,21126.111710
4,s12,19176.463675
5,s13,16434.691116
6,s4,14239.073899
7,s3,11271.558809
8,cycle,8061.057015
9,s14,7339.442015


In [34]:
import plotly.express as px
fig = px.bar(
    variance_df.sort_values("Variance"),
    x="Variance",
    y="Feature",
    orientation="h",
    title="Feature Variance Analysis",
    text="Variance",
    color="Variance",
    color_continuous_scale="Viridis"
)

fig.update_traces(
    texttemplate="%{text:.2f}",
    textposition="outside"
)

fig.update_layout(
    template="plotly_white",
    height=800,
    xaxis_title="Variance",
    yaxis_title="Features"
)

fig.show()

In [19]:
constant_features = variance_df[variance_df["Variance"] == 0]

constant_features

,Feature,Variance


In [20]:
low_variance = variance_df[variance_df["Variance"] < 0.01]

low_variance

,Feature,Variance
24,s16,0.000022


### Observation

- Most features exhibit sufficient variance and contain useful information.
- Only sensor **s16** shows extremely low variance (0.000022).
- The feature is retained at this stage because low variance alone is not sufficient evidence for removal.
- Feature importance analysis during model training will be used to determine whether s16 contributes meaningfully to RUL prediction.

## Step 6: Feature Correlation Analysis

Correlation analysis measures the strength and direction of the relationship between each feature and the target variable (RUL).

- Positive correlation indicates that the feature tends to increase as RUL increases.
- Negative correlation indicates that the feature tends to decrease as RUL increases.
- Features with correlation values close to zero have little linear relationship with RUL.

This analysis helps identify the most informative features for predictive maintenance.

In [21]:
# Calculate correlation of all features with RUL
corr_with_rul = (
    train.corr(numeric_only=True)["RUL"]
    .drop("RUL")
    .sort_values(ascending=False)
)
corr_with_rul

s20         0.002812
s21         0.002791
s18         0.002765
setting3    0.002303
s19         0.002303
s8          0.002086
s1          0.001889
s5          0.001679
s13         0.001501
s6          0.001349
s7         -0.001429
s12        -0.001639
setting2   -0.002280
setting1   -0.002380
unit       -0.003656
s15        -0.003957
s2         -0.004443
s10        -0.008924
s9         -0.024727
s3         -0.032924
s17        -0.032939
s4         -0.045881
s16        -0.053804
s11        -0.056639
s14        -0.078126
cycle      -0.610620
Name: RUL, dtype: float64

In [23]:
corr_df = corr_with_rul.reset_index()
corr_df.columns = ["Feature", "Correlation"]

corr_df.head(25)

,Feature,Correlation
0,s20,0.002812
1,s21,0.002791
2,s18,0.002765
3,setting3,0.002303
4,s19,0.002303
5,s8,0.002086
6,s1,0.001889
7,s5,0.001679
8,s13,0.001501
9,s6,0.001349


In [27]:
import plotly.express as px

corr_df = (
    train.corr(numeric_only=True)["RUL"]
    .drop("RUL")
    .reset_index()
)

corr_df.columns = ["Feature", "Correlation"]
corr_df["AbsCorrelation"] = corr_df["Correlation"].abs()

corr_df = corr_df.sort_values("AbsCorrelation")

fig = px.bar(
    corr_df,
    x="AbsCorrelation",
    y="Feature",
    orientation="h",
    color="AbsCorrelation",
    color_continuous_scale="Viridis",
    text="Correlation"
)

fig.update_traces(
    texttemplate="%{text:.3f}",
    textposition="outside"
)

fig.update_layout(
    title="Feature Importance by Correlation with RUL",
    title_x=0.5,
    template="plotly_white",
    height=850,
    width=1100,
    xaxis_title="Absolute Correlation",
    yaxis_title="Features",
    coloraxis_colorbar_title="|Correlation|"
)

fig.show()

In [28]:
corr_df.sort_values("Correlation", ascending=False).head(10)

,Feature,Correlation,AbsCorrelation
24,s20,0.002812,0.002812
25,s21,0.002791,0.002791
22,s18,0.002765,0.002765
4,setting3,0.002303,0.002303
23,s19,0.002303,0.002303
12,s8,0.002086,0.002086
5,s1,0.001889,0.001889
9,s5,0.001679,0.001679
17,s13,0.001501,0.001501
10,s6,0.001349,0.001349


In [29]:
corr_df.sort_values("Correlation").head(10)


,Feature,Correlation,AbsCorrelation
1,cycle,-0.610620,0.610620
18,s14,-0.078126,0.078126
15,s11,-0.056639,0.056639
20,s16,-0.053804,0.053804
8,s4,-0.045881,0.045881
21,s17,-0.032939,0.032939
7,s3,-0.032924,0.032924
13,s9,-0.024727,0.024727
14,s10,-0.008924,0.008924
6,s2,-0.004443,0.004443


### Feature Scaling

Machine learning algorithms often perform better when numerical features are on a similar scale.

StandardScaler standardizes each feature by removing the mean and scaling to unit variance.

Although tree-based models such as Random Forest and XGBoost do not require feature scaling, a scaled dataset is prepared to support experiments with algorithms that are sensitive to feature magnitudes, such as Linear Regression, Support Vector Regression, and Neural Networks.

In [30]:
from sklearn.preprocessing import StandardScaler
# Initialize scaler
scaler = StandardScaler()
# Scale input features
X_scaled = scaler.fit_transform(X)

# Convert back to DataFrame
X_scaled = pd.DataFrame(
    X_scaled,
    columns=X.columns,
    index=X.index
)

X_scaled.head()

,cycle,setting1,setting2,setting3,s1,s2,s3,s4,s5,s6,...,s12,s13,s14,s15,s16,s17,s18,s19,s20,s21
0,-1.484824,1.218156,0.864668,0.418783,-1.054690,-0.796416,-0.701412,-0.745729,-1.137677,-1.081831,...,-0.989007,0.417814,0.081921,0.063831,-0.694278,-0.638665,-0.114203,0.418783,-1.030999,-1.031756
1,-1.473686,-0.270478,0.414718,0.418783,0.692508,0.713666,0.562449,0.298212,0.363906,0.371152,...,0.331131,0.415786,-0.253086,-0.125677,-0.694278,0.476120,0.655708,0.418783,0.352814,0.358264
2,-1.462548,1.218082,0.867565,0.418783,-1.054690,-0.815965,-0.704332,-0.711202,-1.137677,-1.083668,...,-0.990162,0.417658,-0.013912,0.153387,-0.694278,-0.674626,-0.114203,0.418783,-1.045089,-1.022649
3,-1.451410,1.217824,0.864668,0.418783,-1.054690,-0.822660,-0.722040,-0.702990,-1.137677,-1.081831,...,-0.988862,0.418048,0.096162,0.068362,-0.694278,-0.710586,-0.114203,0.418783,-1.039051,-1.023269
4,-1.440272,0.068094,0.158844,-2.387873,-0.391216,-1.160079,-1.532181,-1.410627,-0.270955,-0.475656,...,-0.741097,-2.389666,-2.358027,2.066982,-0.694278,-1.537685,-2.155843,-2.387873,-0.687814,-0.644612


In [31]:
X_scaled.describe().round(2)

,cycle,setting1,setting2,setting3,s1,s2,s3,s4,s5,s6,...,s12,s13,s14,s15,s16,s17,s18,s19,s20,s21
count,61249.00,61249.00,61249.00,61249.00,61249.00,61249.00,61249.00,61249.00,61249.00,61249.00,...,61249.00,61249.00,61249.00,61249.00,61249.00,61249.00,61249.00,61249.00,61249.00,61249.00
mean,-0.00,0.00,-0.00,0.00,0.00,0.00,-0.00,-0.00,0.00,0.00,...,0.00,-0.00,0.00,-0.00,0.00,0.00,0.00,-0.00,0.00,-0.00
std,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,...,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00
min,-1.48,-1.62,-1.84,-2.39,-1.05,-1.18,-1.65,-1.49,-1.14,-1.09,...,-1.00,-2.39,-2.59,-1.48,-0.69,-1.65,-2.16,-2.39,-1.08,-1.08
25%,-0.81,-0.95,-1.03,0.42,-1.05,-0.81,-0.63,-0.69,-1.14,-1.08,...,-0.95,0.42,-0.06,-0.85,-0.69,-0.64,-0.11,0.42,-1.00,-1.00
50%,-0.13,0.07,0.41,0.42,-0.39,-0.63,-0.47,-0.54,-0.27,-0.47,...,-0.60,0.42,0.19,-0.04,-0.69,-0.49,-0.04,0.42,-0.60,-0.60
75%,0.63,1.22,0.86,0.42,0.69,0.74,0.75,0.84,0.69,0.71,...,0.76,0.42,0.71,0.11,1.44,0.73,0.66,0.42,0.77,0.77
max,4.55,1.22,0.87,0.42,1.73,1.74,1.84,2.00,1.82,1.84,...,1.96,0.44,2.26,2.37,1.44,1.84,1.10,0.42,1.91,1.91


## Save Intermediate Dataset

The dataset containing the generated RUL target is saved for reuse in later stages of the project.

This avoids recalculating the target variable each time preprocessing is performed.

In [32]:
# Combine scaled features with target
train_processed = X_scaled.copy()
train_processed["RUL"] = y

# Save processed dataset
train_processed.to_csv(r"D:\Project\aircraft maintenance system\data\processed\train_processed.csv",
    index=False
)

print("train_processed.csv saved successfully!")

train_processed.csv saved successfully!
